In [1]:
import os
import sys
import subprocess
import glob

# 🎛️ SET THIS TO TRUE FOR TPU, FALSE FOR GPU
FORCE_TPU = True

def repair_environment():

    if FORCE_TPU:
        print("🔍 Starting High-Speed TPU Repair...")

        # 1. Faster Uninstallation
        print("🧹 Wiping libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torch_xla", "torchvision", "numpy", "tensorflow", "huggingface_hub"],
                       capture_output=True)

        # 2. Parallel/Bulk Installation
        print("📥 Installing Synced TPU Stack...")
        common_args = ["install", "-q", "--no-warn-script-location"]

        if FORCE_TPU or glob.glob("/dev/accel*"):
            cmd = [
                sys.executable, "-m", "pip", *common_args,
                "torch==2.8.0",
                "torchvision==0.23.0",
                "torch_xla[tpu]==2.8.0",
                "numpy", "pyarrow", "fsspec",
                "protobuf>=5.28.0",
                "datasets", "transformers", "huggingface_hub>=0.28.0", "wandb",
                "cloud-tpu-client", "scikit-learn", "pandas<3.0.0",
                "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
                "--extra-index-url", "https://download.pytorch.org/whl/cpu"
            ]
            subprocess.check_call(cmd)
        else:
            # Fallback
            cmd = [
                sys.executable, "-m", "pip", *common_args, "-U",
                "torch", "datasets", "pyarrow", "transformers", "huggingface_hub>=0.28.0", "fsspec", "wandb", "scipy", "numpy", "pandas<3.0.0"
            ]
            subprocess.check_call(cmd)

        print("\n✅ TPU REPAIR COMPLETE.")
        print("⚠️ Click 'Run' -> 'Restart Session' NOW.")

    else:
        print("🔍 Starting Robust GPU Repair...")

        # 1. Clean Wipe
        print("🧹 Wiping conflicting libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torchvision", "torchaudio", "huggingface_hub"],
                       capture_output=True)

        # 2. Setup Arguments
        common_args = ["install", "-q", "--no-warn-script-location"]

        try:
            print("📥 Installing GPU/CUDA Stack...")

            print("   ⚡ Part 1: PyTorch Core...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args,
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"
            ])

            print("   ⚡ Part 2: Transformers & Data...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args, "-U",
                "datasets", "transformers", "huggingface_hub>=0.28.0",
                "wandb", "pandas<3.0.0"
            ])

            print("\n✅ GPU REPAIR COMPLETE.")
            print("⚠️ MANDATORY: Click 'Run' -> 'Restart Session' NOW.")

        except subprocess.CalledProcessError as e:
            print(f"\n❌ Installation failed. Error: {e}")
            print("💡 Try manually restarting the session and running this cell again.")

if __name__ == "__main__":
    repair_environment()

🔍 Starting High-Speed TPU Repair...
🧹 Wiping libraries...
📥 Installing Synced TPU Stack...

✅ TPU REPAIR COMPLETE.
⚠️ Click 'Run' -> 'Restart Session' NOW.


In [1]:
import torch.nn as nn
from transformers import AutoModel

class RegressionModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.modernBert = AutoModel.from_pretrained(model_name)
        self.regression_head = nn.Sequential(
            nn.Linear(768, 256),
            nn.LayerNorm(256),
            nn.Tanh(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.modernBert(input_ids=input_ids, attention_mask=attention_mask)

        # [batch_size, seq_len, hidden_size]
        last_hidden = outputs.last_hidden_state

        # Adds extra last dimension and broadcast to all hidden dims
        # [batch_size, seq_len] -> [batch_size, seq_len, 1] -> [batch_size, seq_len, hidden_size]
        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()

        # Sum the logits along token (seq_len) dimension
        sum_embeddings = torch.sum(last_hidden * mask, 1)

        # Sum total logits
        sum_mask = mask.sum(1)

        # Find Average
        mean_pooled = sum_embeddings / sum_mask

        # Pass new logits to regression head
        logits = self.regression_head(mean_pooled)

        # # Sigmoid
        # logits = torch.sigmoid(logits)

        return {"logits": logits}

    def save_pretrained(self, save_directory):
        os.makedirs(save_directory, exist_ok=True)
        self.modernBert.config.save_pretrained(save_directory)
        unified_weights_path = os.path.join(save_directory, "model.pt")
        torch.save(self.state_dict(), unified_weights_path)
        print(f"🔥 Unified weights securely saved to {unified_weights_path}")

In [ ]:
# Imports (Manifesting that my loss curve will look like this)
import io
import os
import sys
import time
import json
import site
import torch
import importlib
import numpy as np
import pyarrow.parquet as pq
from torch.utils.data import DataLoader
from datasets import load_dataset, Dataset
from huggingface_hub import hf_hub_download, create_repo, HfApi
import datasets
import torch.nn as nn
import torch_xla # Added explicit import for torch_xla
import torch_xla.core.xla_model as xm
from transformers import AutoModelForSequenceClassification
from transformers import get_scheduler
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np
# Ensure PJRT runtime gets selected, not XRT
for key in ["XRT_TPU_CONFIG", "PJRT_SELECT_DEVICE", "TPU_PROCESS_ADDRESSES"]:
    os.environ.pop(key, None)
os.environ["PJRT_DEVICE"] = "TPU"
# Add the framework quarantine just in case!
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
# Prevent C++ thread deadlocks during 10B token streaming
os.environ["OMP_NUM_THREADS"] = "1"
import pyarrow as pa
pa.set_cpu_count(1)
pa.set_io_thread_count(1)
# Tell everything to stay away from the TPU except PyTorch
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"

# Force Path Refresh
if 'site' in sys.modules:
    importlib.reload(site)

# Get HF_TOKEN and WANDB_API_KEY
def get_secret(key_name):

    # Try Colab
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except:
        pass

    # Try Kaggle
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key_name)
    except:
        pass

    # Local Env
    return os.getenv(key_name)

# Get the float value of something (mainly for losses)
def to_float(x):
    return x.item() if hasattr(x, 'item') else float(x)

# Globals

hf_token = get_secret("HF_TOKEN")
model_repo_id = "JamesResearch1216/ModernBERT-BT_Easiness_v6"
data_repo_id = "JamesResearch1216/BT_Easiness_Data"
num_epochs = 15
num_data_rows = 4251
batch_size = 16


def get_dataloader(is_train: bool):

  dataset = load_dataset(
      path = data_repo_id,
      split = "train" if is_train else "validation",
      token = hf_token
  )

  def collate_fn(batch):
    return {
      "input_ids": torch.tensor([item["input_ids"] for item in batch], dtype=torch.long),
      "attention_mask": torch.tensor([item["attention_mask"] for item in batch], dtype=torch.long),
      "labels": torch.tensor([item["labels"] for item in batch], dtype=torch.float32).unsqueeze(1) # Reshape labels to (batch_size, 1)
    }

  data_loader = DataLoader(
      dataset,
      batch_size = batch_size,
      shuffle = is_train,
      drop_last = True,
      pin_memory = False,
      collate_fn = collate_fn
  )

  return data_loader

def train():

  pre_val_loss = 0

  device = xm.xla_device() if not hasattr(torch_xla, 'device') else torch_xla.device()

  model = RegressionModel("answerdotai/ModernBERT-base").to(device)
  # print(model.modernBert)

  for param in model.modernBert.embeddings.parameters():
      param.requires_grad = False

  for layer in model.modernBert.layers[:10]:
      for param in layer.parameters():
          param.requires_grad = False

  # Now define the optimizer (it will automatically only update unfrozen layers)
  optimizer = torch.optim.AdamW([
      {"params": filter(lambda p: p.requires_grad, model.modernBert.parameters()),
          "lr": 1e-4,
          "weight_decay": 0.01
      },
      {
          "params": model.regression_head.parameters(),
          "lr": 1e-3 ,
          "weight_decay": 0.01
      }
  ])
  optimizer.zero_grad()
  lr_scheduler = get_scheduler(
    name = "cosine",
    optimizer = optimizer,
    num_warmup_steps = int(num_epochs*num_data_rows/(batch_size*10)),
    num_training_steps = int(2*num_epochs*num_data_rows/batch_size)
  )

  train_loader = get_dataloader(is_train = True)
  validation_loader = get_dataloader(is_train = False)

  loss_fct = nn.MSELoss(reduction='none')


  # Epoch
  best_val_loss = float('inf')
  for i in range(0, num_epochs):

    print(f"Epoch {i}")
    model.train()

    for step, batch in enumerate(train_loader):

      input_ids = batch["input_ids"].to(device)
      labels = batch["labels"].to(device)
      attention_mask = batch["attention_mask"].to(device)

      outputs = model(input_ids=input_ids, attention_mask=attention_mask)
      preds = outputs["logits"].squeeze()
      flat_labels = labels.squeeze()
      raw_loss = loss_fct(preds, flat_labels)
      distance_from_center = torch.abs(flat_labels - 0.5)
      weights = 1.0 + 6.0 * distance_from_center  # tune the multiplier
      loss = (raw_loss * weights).mean()

      loss.backward()

      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
      xm.optimizer_step(optimizer)
      xm.mark_step()
      lr_scheduler.step()
      optimizer.zero_grad()
      if (step %10) == 0:
        print(f"\tStep {step} Loss: {loss.item()}")

    # This means to force training steps to finalize smoothly
    # Prevents overhang
    torch_xla.sync()

    model.eval()
    all_preds = []
    all_targets = []
    total_validation_loss = 0

    with torch.no_grad():
      for step, batch in enumerate(validation_loader):

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids, attention_mask = attention_mask
        )

        flat_logits = outputs["logits"].squeeze()
        flat_labels = labels.squeeze()

        val_preds = flat_logits.clamp(0, 1)
        val_loss = nn.functional.mse_loss(val_preds, flat_labels)

        # Do this to compile a static graph for eval
        xm.mark_step()

        preds = val_preds.detach().cpu().numpy()  # <--- MUST APPLY SIGMOID HERE
        targets = flat_labels.detach().cpu().numpy()

        total_validation_loss += val_loss.item()
        all_preds.extend(val_preds.detach().cpu().numpy().tolist())
        all_targets.extend(flat_labels.detach().cpu().numpy().tolist())

        if step == 0:
          for i in range(min(5, len(preds))):
            print(f"Sample {i} | True: {targets[i]:.3f} | Pred: {preds[i]:.3f}")
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    avg_val_loss = total_validation_loss / len(validation_loader)
    print(f"\tAverage Validation Loss: {avg_val_loss}")
    print(f"  MAE: {mean_absolute_error(all_targets, all_preds):.4f}")
    print(f"  R²:  {r2_score(all_targets, all_preds):.4f}")

    # Bucketed accuracy to see WHERE it struggles
    for lo, hi in [(0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.01)]:
        mask = (all_targets >= lo) & (all_targets < hi)
        if mask.sum() > 0:
            bucket_mae = mean_absolute_error(all_targets[mask], all_preds[mask])
            print(f"  Range [{lo:.1f}-{hi:.1f}): n={mask.sum()}, MAE={bucket_mae:.4f}")

    if avg_val_loss < best_val_loss:
      best_val_loss = avg_val_loss
      model.save_pretrained("./best_checkpoint")
      print(f"  New best model saved (val_loss: {avg_val_loss:.6f})")


  api = HfApi()

  try:
      api.create_repo(repo_id=model_repo_id, token=hf_token, repo_type="model", exist_ok=True)
      print(f"Repository '{model_repo_id}' ensured on Hugging Face.")
  except Exception as e:
      print(f"Error creating/checking Hugging Face repository: {e}")
      return # Exit if repo creation fails

  model.to("cpu")
  try:
      # Save the model locally first
      output_dir = f"./{model_repo_id}"
      model.save_pretrained(output_dir)

      # Push to Hugging Face Hub
      api.upload_folder(
          folder_path=output_dir,
          repo_id=model_repo_id,
          token=hf_token,
          repo_type="model"
      )
      print(f"Model successfully pushed to Hugging Face Hub: {model_repo_id}")
  except Exception as e:
      print(f"Error pushing model to Hugging Face: {e}")


if __name__ == "__main__":
  train()

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 0


/tmp/ipykernel_2507/594700741.py:171: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


	Step 0 Loss: 0.41264307498931885
	Step 10 Loss: 0.2680073380470276
	Step 20 Loss: 0.2752231955528259
	Step 30 Loss: 0.19243475794792175
	Step 40 Loss: 0.1599339097738266
	Step 50 Loss: 0.20183268189430237
	Step 60 Loss: 0.16416430473327637
	Step 70 Loss: 0.34348586201667786
	Step 80 Loss: 0.05887141823768616
	Step 90 Loss: 0.09752827882766724
	Step 100 Loss: 0.10803678631782532
	Step 110 Loss: 0.0906512439250946
	Step 120 Loss: 0.09606467187404633
	Step 130 Loss: 0.10134229063987732
	Step 140 Loss: 0.15844333171844482
	Step 150 Loss: 0.12463843822479248
	Step 160 Loss: 0.1875452995300293
	Step 170 Loss: 0.2599909007549286
	Step 180 Loss: 0.07224959135055542
	Step 190 Loss: 0.08697357773780823
	Step 200 Loss: 0.06227564811706543
	Step 210 Loss: 0.056738562881946564
	Step 220 Loss: 0.06541533023118973
	Step 230 Loss: 0.06865878403186798
	Step 240 Loss: 0.1483938843011856
	Step 250 Loss: 0.14231105148792267
	Step 260 Loss: 0.13886211812496185


...

Sample 0 | True: 0.235 | Pred: 0.202
Sample 1 | True: 0.731 | Pred: 0.763
Sample 2 | True: 1.000 | Pred: 0.938
Sample 3 | True: 0.533 | Pred: 0.363
Sample 4 | True: 0.470 | Pred: 0.642
	Average Validation Loss: 0.013078764819636428
  MAE: 0.0907
  R²:  0.7483
  Range [0.0-0.2): n=38, MAE=0.1025
  Range [0.2-0.4): n=91, MAE=0.0862
  Range [0.4-0.6): n=142, MAE=0.0902
  Range [0.6-0.8): n=126, MAE=0.0808
  Range [0.8-1.0): n=67, MAE=0.1097
🔥 Unified weights securely saved to ./best_checkpoint/model.pt
  New best model saved (val_loss: 0.013079)
Repository 'JamesResearch1216/ModernBERT-BT_Easiness_v6' ensured on Hugging Face.
🔥 Unified weights securely saved to ./JamesResearch1216/ModernBERT-BT_Easiness_v6/model.pt


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...T-BT_Easiness_v6/model.pt:  52%|#####2    |  312MB /  597MB            

Model successfully pushed to Hugging Face Hub: JamesResearch1216/ModernBERT-BT_Easiness_v6


In [ ]:
# 1. Initialize an empty version of your architecture skeleton
trained_model = RegressionModel("answerdotai/ModernBERT-base")

# 2. Load the unified state dictionary weights from the file
state_dict = torch.load("./RoBERTa-BT_Easiness/model.pt")

# 3. Inject the weights directly into the skeleton
trained_model.load_state_dict(state_dict)
trained_model.eval()